# Experimentos — análisis semántico del MCP SISAV2 (Grupo 2)

Mide el aporte de cada componente de las *tools* de análisis (`buscar_iniciativas_similares`, `detectar_duplicados`): **embeddings + similitud coseno**, **semántico vs. keyword (TF-IDF)**, efecto de **k** y del **umbral** de duplicados.

> **Corpus.** Ilustrativo: objetivos de iniciativas VcM redactados a mano, representativos de las modalidades reales (EA/VEDP/VCT). La cohorte real se construye desde SISAV2 en la demo (`sisav2-mcp index-demo`) y se excluye de git por PII. El corpus golden del curso (buenas 2606/2648/2650 · malas 2690/2724/2788) llega como PDFs; aquí usamos objetivos cortos etiquetados por tema para poder medir precisión.

**Stack:** `sentence-transformers` (paraphrase-multilingual-MiniLM-L12-v2), `scikit-learn` (TF-IDF baseline), `numpy`. Reproducible: `uv run` + este notebook.

In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# id, tema (ground-truth), objetivo
CORPUS = [
    ('E1', 'educacion', 'Realizar talleres de robotica educativa para estudiantes de ensenanza media de colegios municipales de la comuna.'),
    ('E2', 'educacion', 'Ejecutar talleres de robotica dirigidos a alumnos de educacion media en establecimientos municipales del sector.'),
    ('E3', 'educacion', 'Dictar charlas vocacionales de ingenieria a estudiantes de cuarto medio para acercarlos a la vida universitaria.'),
    ('S1', 'salud', 'Operativo de salud preventiva con controles y educacion en autocuidado para adultos mayores de juntas de vecinos.'),
    ('S2', 'salud', 'Jornadas de promocion de salud bucal y entrega de kits de higiene en centros comunitarios de la comuna.'),
    ('M1', 'medioambiente', 'Programa de reciclaje y compostaje domiciliario con capacitacion a familias sobre manejo de residuos organicos.'),
    ('M2', 'medioambiente', 'Restauracion de un humedal urbano mediante reforestacion con especies nativas y monitoreo de biodiversidad.'),
    ('C1', 'cultura', 'Ciclo de conciertos didacticos de musica docta en espacios publicos para difundir el patrimonio cultural.'),
    ('P1', 'emprendimiento', 'Asesoria en formalizacion y contabilidad basica para microemprendedores de ferias libres de la region.'),
    ('P2', 'emprendimiento', 'Capacitacion en finanzas y tributacion simple para pequenos comerciantes de mercados populares.'),
]
ids = [c[0] for c in CORPUS]
temas = [c[1] for c in CORPUS]
textos = [c[2] for c in CORPUS]
print(f'{len(CORPUS)} iniciativas, {len(set(temas))} temas: {sorted(set(temas))}')

10 iniciativas, 5 temas: ['cultura', 'educacion', 'emprendimiento', 'medioambiente', 'salud']


## 1. Embeddings + similitud coseno
Un *embedding* proyecta cada objetivo a un vector denso; la **similitud coseno** (coseno del ángulo entre vectores) mide cercanía **semántica** independiente del largo. Esperamos que pares del **mismo tema** tengan coseno mayor que pares de temas distintos.

In [2]:
from sentence_transformers import SentenceTransformer

MODELO = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
model = SentenceTransformer(MODELO)
emb = model.encode(textos, normalize_embeddings=True)
S = cosine_similarity(emb)
print('dim embedding:', emb.shape[1])

intra, inter = [], []
for i in range(len(ids)):
    for j in range(i + 1, len(ids)):
        (intra if temas[i] == temas[j] else inter).append(S[i, j])
print(f'coseno medio MISMO tema : {np.mean(intra):.3f}')
print(f'coseno medio OTRO  tema : {np.mean(inter):.3f}')
print(f'separacion (intra-inter): {np.mean(intra) - np.mean(inter):.3f}')

C:\Users\welin\Desktop\universidad\NLP\Proyecto A + S\repos\sisav2-mcp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5142.93it/s]

dim embedding: 384
coseno medio MISMO tema : 0.506
coseno medio OTRO  tema : 0.206
separacion (intra-inter): 0.300


In [3]:
# Vecino mas cercano de cada iniciativa (top-1)
for i, cid in enumerate(ids):
    orden = np.argsort(-S[i])
    j = [k for k in orden if k != i][0]
    marca = 'OK ' if temas[i] == temas[j] else 'x  '
    print(f'{marca}{cid} ({temas[i]:14}) -> {ids[j]} ({temas[j]:14}) cos={S[i, j]:.3f}')

OK E1 (educacion     ) -> E2 (educacion     ) cos=0.848
OK E2 (educacion     ) -> E1 (educacion     ) cos=0.848
OK E3 (educacion     ) -> E1 (educacion     ) cos=0.395
OK S1 (salud         ) -> S2 (salud         ) cos=0.532
OK S2 (salud         ) -> S1 (salud         ) cos=0.532
x  M1 (medioambiente ) -> S2 (salud         ) cos=0.506
OK M2 (medioambiente ) -> M1 (medioambiente ) cos=0.367
x  C1 (cultura       ) -> S2 (salud         ) cos=0.287
OK P1 (emprendimiento) -> P2 (emprendimiento) cos=0.543
OK P2 (emprendimiento) -> P1 (emprendimiento) cos=0.543


## 2. Búsqueda semántica vs. keyword (TF-IDF)
Consulta que **parafrasea** una iniciativa usando otras palabras. La búsqueda por **keyword** (TF-IDF: pondera coincidencia de términos) falla si no comparten vocabulario; la **semántica** captura el significado. Es la diferencia que exige explicar el Q&A.

In [4]:
consulta = 'orientacion para que jovenes escolares descubran carreras del area tecnologica'
objetivo_real = 'E3'  # charlas vocacionales de ingenieria a cuarto medio

# --- semantico ---
q_emb = model.encode([consulta], normalize_embeddings=True)
sem = cosine_similarity(q_emb, emb)[0]
rank_sem = [ids[k] for k in np.argsort(-sem)]

# --- keyword (TF-IDF) ---
vec = TfidfVectorizer()
M = vec.fit_transform(textos + [consulta])
kw = cosine_similarity(M[-1], M[:-1])[0]
rank_kw = [ids[k] for k in np.argsort(-kw)]

print('consulta:', consulta)
print('objetivo esperado:', objetivo_real, '(E3)')
print(f'  SEMANTICO  top-3: {rank_sem[:3]}  -> rank de E3: {rank_sem.index(objetivo_real)+1}')
print(f'  KEYWORD    top-3: {rank_kw[:3]}  -> rank de E3: {rank_kw.index(objetivo_real)+1}')

consulta: orientacion para que jovenes escolares descubran carreras del area tecnologica
objetivo esperado: E3 (E3)
  SEMANTICO  top-3: ['E3', 'E2', 'E1']  -> rank de E3: 1
  KEYWORD    top-3: ['E2', 'P2', 'P1']  -> rank de E3: 4


## 3. Efecto de k (precisión@k)
Para cada iniciativa como consulta (leave-one-out), recuperamos las top-*k* y medimos qué fracción es del **mismo tema**. Subir *k* recupera más, pero suele **bajar la precisión** (entran vecinos de otros temas).

In [5]:
def precision_at_k(k):
    ps = []
    for i in range(len(ids)):
        orden = [j for j in np.argsort(-S[i]) if j != i][:k]
        aciertos = sum(1 for j in orden if temas[j] == temas[i])
        ps.append(aciertos / k)
    return np.mean(ps)

for k in (1, 3, 5):
    print(f'precision@{k} = {precision_at_k(k):.3f}')

precision@1 = 0.800
precision@3 = 0.400
precision@5 = 0.240


## 4. Umbral de `detectar_duplicados`
Se marcan como candidatos los pares con coseno ≥ umbral. Umbral **alto** = pocos candidatos, casi todos reales (alta precisión, baja cobertura); umbral **bajo** = más candidatos con más falsos positivos. `detectar_duplicados` usa 0.85 por defecto: candidatos para **revisión humana**, no un veredicto.

In [6]:
pares = [(i, j) for i in range(len(ids)) for j in range(i + 1, len(ids))]
for umbral in (0.75, 0.85, 0.90):
    marcados = [(i, j) for (i, j) in pares if S[i, j] >= umbral]
    tp = [(i, j) for (i, j) in marcados if temas[i] == temas[j]]
    fp = [(i, j) for (i, j) in marcados if temas[i] != temas[j]]
    ej = ', '.join(f'{ids[i]}~{ids[j]}({S[i,j]:.2f})' for i, j in marcados) or '-'
    print(f'umbral {umbral:.2f}: {len(marcados)} candidatos | mismo-tema={len(tp)} otro-tema(FP)={len(fp)} | {ej}')

umbral 0.75: 1 candidatos | mismo-tema=1 otro-tema(FP)=0 | E1~E2(0.85)
umbral 0.85: 0 candidatos | mismo-tema=0 otro-tema(FP)=0 | -
umbral 0.90: 0 candidatos | mismo-tema=0 otro-tema(FP)=0 | -


## 5. Hallazgos y límites

- **Los embeddings separan por significado:** el coseno medio intra-tema supera al inter-tema, y el vecino más cercano suele ser del mismo tema.
- **Semántico > keyword** cuando la consulta parafrasea sin compartir palabras: TF-IDF hunde el objetivo correcto; los embeddings lo rankean arriba. Por eso las tools usan embeddings y no `LIKE`/keyword.
- **k es un trade-off** recuperación↔precisión (0.80 → 0.40 → 0.24 al pasar de k=1 a 5): más resultados, menos precisión. k=5 por defecto para revisión humana.
- **El umbral es sensible.** El par casi-duplicado E1~E2 (coseno **0.848**) queda *justo bajo* 0.85: a 0.75 se detecta sin falsos positivos, a 0.85 se pierde. Ilustra por qué el umbral se calibra y el resultado va a **revisión humana**, nunca como decisión automática.
- **Corpus pequeño:** con pocas iniciativas las métricas son inestables y sensibles a cada caso; la cohorte real (25–50) da estimaciones más firmes.
- **Determinismo:** el *embedder* es determinista (mismo texto → mismo vector); el no-determinismo relevante está en el **LLM cliente**, no en esta capa. Distinto de las tools de escritura, aquí no hay mutación: es apoyo a la decisión humana.